# 02 — Data Cleaning
**Projeto:** iFood Customer Analytics (simulação)  
**Squad:** Customer Intelligence · **Demanda:** MKT-001

## Objetivo
Aplicar as regras de tratamento definidas no Data Profiling, preservar a fonte original e disponibilizar uma base analítica validada.

**Entrada:** `data/raw/mkt_data.csv`  
**Saídas:** `data/processed/mkt_data_clean.parquet` e `reports/data_cleaning_report.csv`.

**Escopo:** tratamento, validação e persistência; estatística e análises de negócio ficam para os próximos notebooks. Execute a partir da pasta `notebooks/`. Dependências: `pandas`, `pyarrow` (instale no ambiente do kernel).

## 1. Configuração e carregamento

In [14]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path('../data/raw/mkt_data.csv')
PROCESSED_PATH = Path('../data/processed')
REPORTS_PATH = Path('../reports')
OUTPUT_PATH = PROCESSED_PATH / 'mkt_data_clean.parquet'

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

In [15]:
df = pd.read_csv(RAW_PATH)
initial_rows, initial_columns = df.shape
print(f'Registros: {initial_rows} | Colunas: {initial_columns}')
df.head()

Registros: 2205 | Colunas: 44


,Unnamed: 0,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,...,education_Graduation,education_Master,education_PhD,MntTotal,MntRegularProds,AcceptedCmpOverall,marital_status,education_level,kids,expenses
0,0,58138.0,0,0,58,635,88,546,172,88,...,3.0,NaN,NaN,1529,1441,0,Single,Graduation,0,1529
1,1,46344.0,1,1,38,11,1,6,2,1,...,3.0,NaN,NaN,21,15,0,Single,Graduation,2,21
2,2,71613.0,0,0,26,426,49,127,111,21,...,3.0,NaN,NaN,734,692,0,Together,Graduation,0,734
3,3,26646.0,1,0,26,11,4,20,10,3,...,3.0,NaN,NaN,48,43,0,Together,Graduation,1,48
4,4,58293.0,1,0,94,173,43,118,46,27,...,NaN,NaN,5.0,407,392,0,Married,PhD,1,407


In [16]:
# Preservar a referência original e trabalhar exclusivamente na cópia.
df_clean = df.copy()

## 2. Índice técnico e duplicidades
O índice exportado não identifica clientes de maneira confiável. Para este desafio, registros integralmente iguais sem esse índice são considerados duplicatas. Em produção, a granularidade e a regra exigiriam validação com o proprietário da fonte.

In [17]:
df_clean = df_clean.drop(columns=['Unnamed: 0'])
duplicates_count = int(df_clean.duplicated().sum())
print(f'Duplicatas por conteúdo: {duplicates_count}')
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
assert not df_clean.duplicated().any()
df_clean.shape

Duplicatas por conteúdo: 184


(2021, 43)

## 3. Nulos estruturais
As colunas auxiliares codificam categorias já presentes em `marital_status` e `education_level`. Validamos a correspondência antes de removê-las; não imputamos valores nem descartamos clientes.

In [18]:
marital_columns = df_clean.filter(regex='^marital_').drop(columns=['marital_status']).columns.tolist()
education_columns = df_clean.filter(regex='^education_').drop(columns=['education_level']).columns.tolist()
structural_columns = marital_columns + education_columns

for columns, prefix, source in [(marital_columns, 'marital_', 'marital_status'),
                                (education_columns, 'education_', 'education_level')]:
    for col in columns:
        category = col[len(prefix):]
        assert (df_clean[col].notna() == df_clean[source].eq(category)).all(), col
        assert df_clean[col].nunique() == 1, col
        print(f'{col}: OK')

marital_Divorced: OK
marital_Married: OK
marital_Single: OK
marital_Together: OK
marital_Widow: OK
education_2n Cycle: OK
education_Basic: OK
education_Graduation: OK
education_Master: OK
education_PhD: OK


In [19]:
df_clean = df_clean.drop(columns=structural_columns)
remaining_nulls = int(df_clean.isna().sum().sum())
assert remaining_nulls == 0, 'Nulos não tratados.'
print(f'Nulos restantes: {remaining_nulls}')
df_clean.shape

Nulos restantes: 0


(2021, 33)

## 4. Colunas constantes
Removemos apenas as duas colunas previstas no diagnóstico. Outros campos constantes inesperados exigem revisão.

In [20]:
constant_columns = ['Z_CostContact', 'Z_Revenue']
assert all(df_clean[col].nunique(dropna=False) == 1 for col in constant_columns)
df_clean = df_clean.drop(columns=constant_columns)
df_clean.shape

(2021, 31)

## 5. Inconsistências remanescentes
Preservamos valores negativos em `MntRegularProds` até confirmar a regra de negócio. Não podemos presumir que sejam erros ou estornos.

In [21]:
negative_records = df_clean.loc[df_clean['MntRegularProds'] < 0]
negative_count = len(negative_records)
print(f'Registros negativos preservados: {negative_count}')
negative_records

Registros negativos preservados: 3


,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,...,Response,Age,Customer_Days,MntTotal,MntRegularProds,AcceptedCmpOverall,marital_status,education_level,kids,expenses
1248,6560.0,0,0,2,67,11,26,4,3,262,...,0,38,2358,111,-151,0,Single,Master,0,111
1649,7144.0,0,2,92,81,4,33,5,2,291,...,0,54,2363,125,-166,0,Single,PhD,2,125
1800,4428.0,0,1,0,16,4,12,2,4,321,...,0,51,2426,38,-283,0,Married,Graduation,1,38


## 6. Validação de integridade
Os testes abaixo impedem a exportação se as regras esperadas não forem atendidas. Mantemos as variáveis redundantes conforme decisão anterior.

In [22]:
assert not df_clean.duplicated().any(), 'Duplicatas remanescentes.'
assert not df_clean.isna().any().any(), 'Nulos remanescentes.'
assert (df_clean['kids'] == df_clean['Kidhome'] + df_clean['Teenhome']).all()
assert (df_clean['expenses'] == df_clean['MntTotal']).all()
assert len(df_clean) == initial_rows - duplicates_count
assert df_clean.shape[1] == initial_columns - 1 - len(structural_columns) - len(constant_columns)
print('Testes de integridade aprovados.')
final_rows, final_columns = df_clean.shape
print(f'Dimensões: {df.shape} -> {df_clean.shape}')

Testes de integridade aprovados.
Dimensões: (2205, 44) -> (2021, 31)


## 7. Relatório de tratamento

In [23]:
metrics = {
    'initial_rows': initial_rows,
    'final_rows': final_rows,
    'removed_rows': initial_rows - final_rows,
    'initial_columns': initial_columns,
    'final_columns': final_columns,
    'removed_columns': initial_columns - final_columns,
    'duplicates_removed': duplicates_count,
    'technical_columns_removed': 1,
    'structural_columns_removed': len(structural_columns),
    'constant_columns_removed': len(constant_columns),
    'remaining_duplicates': int(df_clean.duplicated().sum()),
    'remaining_nulls': remaining_nulls,
    'remaining_negative_records': negative_count
}
cleaning_report = pd.DataFrame(metrics.items(), columns=['metric', 'value'])
cleaning_report

,metric,value
0,initial_rows,2205
1,final_rows,2021
2,removed_rows,184
3,initial_columns,44
4,final_columns,31
5,removed_columns,13
6,duplicates_removed,184
7,technical_columns_removed,1
8,structural_columns_removed,10
9,constant_columns_removed,2


## 8. Persistência e validação
O arquivo Parquet preserva os tipos de dados e permite que os próximos notebooks sejam independentes. Se necessário, instale `pyarrow` no kernel com `%pip install pyarrow` e execute novamente.

In [24]:
df_clean.to_parquet(OUTPUT_PATH, index=False, engine='pyarrow')
print(f'Arquivo salvo: {OUTPUT_PATH}')

Arquivo salvo: ..\data\processed\mkt_data_clean.parquet


In [25]:
df_validation = pd.read_parquet(OUTPUT_PATH, engine='pyarrow')
pd.testing.assert_frame_equal(df_clean, df_validation)
print('Arquivo Parquet validado com sucesso.')

Arquivo Parquet validado com sucesso.


In [26]:
# Publicar o relatório após validar a persistência.
cleaning_report.to_csv(REPORTS_PATH / 'data_cleaning_report.csv', index=False)
print('Relatório exportado.')

Relatório exportado.


# 9. Conclusões e parecer técnico

## 9.1. Resumo executivo

A etapa de Data Cleaning da demanda MKT-001 teve como objetivo disponibilizar uma base analítica consistente e reproduzível para as investigações conduzidas pela squad de Customer Intelligence, atendendo às necessidades da área de Marketing & CRM.

As transformações foram fundamentadas nas evidências identificadas durante a etapa de Data Profiling, priorizando a preservação das informações relevantes para o negócio e a rastreabilidade das decisões técnicas.

O tratamento resultou na redução de 2.205 para 2.021 registros e de 44 para 31 variáveis, sem alterações na fonte original.

## 9.2. Indicadores consolidados

| Indicador | Antes | Depois |
|---|---:|---:|
| Total de registros | 2.205 | 2.021 |
| Total de variáveis | 44 | 31 |
| Duplicidades por conteúdo | 184 | 0 |
| Colunas com valores nulos | 10 | 0 |
| Colunas constantes | 2 | 0 |
| Índices técnicos desnecessários | 1 | 0 |
| Registros negativos em MntRegularProds | 3 | 3 |

### Indicadores de impacto

- Redução do volume de registros: 8,34%.
- Redução do número de variáveis: 29,55%.
- Taxa de preservação dos registros: 91,66%.
- Eliminação das duplicidades identificadas: 100%.
- Eliminação dos valores nulos identificados: 100%, mediante remoção das colunas auxiliares.

## 9.3. Registro das decisões técnicas

| Decisão | Justificativa | Risco residual |
|---|---|---|
| Remoção de Unnamed: 0 | Índice técnico sem finalidade analítica | Baixo |
| Remoção de duplicidades | Evitar contabilização repetida de registros equivalentes | Exclusão de registros legítimos sem identificador único |
| Remoção de colunas auxiliares | Informação categórica disponível nas variáveis originais | Necessidade de recriação para determinados modelos |
| Remoção de colunas constantes | Ausência de variabilidade na amostra | Possível relevância em bases futuras |
| Preservação dos valores negativos | Ausência de confirmação da regra de negócio | Possível impacto em análises financeiras |
| Preservação de variáveis redundantes | Compatibilidade com requisitos analíticos e nomenclatura original | Risco de dupla contabilização em análises futuras |

A remoção de registros duplicados por conteúdo constitui uma premissa adotada para este projeto. Em ambiente produtivo, essa decisão deverá ser validada com o responsável pela fonte de dados.

## 9.4. Integridade e qualidade dos dados

Foram implementados testes automatizados para verificar:

- Ausência de duplicidades após o tratamento.
- Ausência de valores nulos remanescentes.
- Consistência entre `kids`, `Kidhome` e `Teenhome`.
- Equivalência entre `expenses` e `MntTotal`.
- Correspondência entre o número de registros removidos e as duplicidades identificadas.
- Integridade dos dados após persistência e releitura do arquivo Parquet.

A aprovação da persistência depende da execução bem-sucedida do teste de equivalência entre o DataFrame tratado e o arquivo exportado.

## 9.5. Riscos e limitações

### RISK-001 — Valores financeiros negativos

**Variável:** MntRegularProds

**Quantidade:** 3 registros.

**Incidência:** aproximadamente 0,15% da base tratada.

**Descrição:** existência de valores negativos em uma variável relacionada a gastos com produtos.

**Impacto potencial:** alteração de indicadores financeiros, medidas de tendência central e distribuições estatísticas.

**Mitigação:** preservação dos registros até esclarecimento da regra de negócio.

**Responsável sugerido:** proprietário da fonte de dados.

**Status:** pendente de esclarecimento.

### RISK-002 — Redundância entre variáveis

**Variáveis:** expenses e MntTotal.

**Descrição:** equivalência integral entre os valores das duas variáveis na amostra analisada.

**Impacto potencial:** dupla contabilização de gastos ou redundância em modelos estatísticos.

**Mitigação:** utilizar apenas uma das variáveis em agregações financeiras e avaliar sua exclusão em futuros modelos.

**Status:** risco identificado e documentado.

### RISK-003 — Critério de deduplicação

**Descrição:** utilização da equivalência integral do conteúdo como critério para identificação de registros duplicados.

**Impacto potencial:** remoção de registros legítimos que apresentem atributos idênticos.

**Mitigação:** validar a granularidade da fonte e identificar uma chave de negócio adequada antes de uma eventual implementação produtiva.

**Status:** premissa do projeto.

## 9.6. Contrato de saída dos dados

| Propriedade | Especificação |
|---|---|
| Arquivo | mkt_data_clean.parquet |
| Localização | data/processed/ |
| Formato | Apache Parquet |
| Total de registros esperado | 2.021 |
| Total de variáveis esperado | 31 |
| Valores nulos esperados | 0 |
| Duplicidades esperadas | 0 |
| Índice técnico | Não persistido |
| Consumidor imediato | 03_exploratory_analysis.ipynb |
| Fonte de origem | data/raw/mkt_data.csv |

Qualquer alteração nas regras de tratamento deverá ser documentada e reproduzida a partir da fonte original.

## 9.7. Critérios de aceite

- [x] Diagnóstico do Data Profiling considerado.
- [x] Regras de tratamento implementadas.
- [x] Duplicidades identificadas removidas.
- [x] Valores nulos estruturais tratados.
- [x] Colunas constantes removidas.
- [x] Inconsistências remanescentes documentadas.
- [x] Relatório de tratamento gerado.
- [ ] Persistência e equivalência do Parquet confirmadas no ambiente de execução.
- [x] Fonte original preservada pela lógica implementada.

## 9.8. Entregáveis

| Entregável | Localização |
|---|---|
| Notebook de tratamento | notebooks/02_data_cleaning.ipynb |
| Base analítica tratada | data/processed/mkt_data_clean.parquet |
| Relatório de tratamento | reports/data_cleaning_report.csv |

## 9.9. Recomendações para as próximas etapas

Durante a análise exploratória,